# Benchmark Results Visualization

This notebook provides interactive visualizations of benchmark results and GPU monitoring data from the `results` directory.

## Features:
- Load all benchmark runs from the results directory
- Interactive filtering by model, run, and task
- GPU utilization time-series graphs
- Performance metrics and statistics
- Task comparison visualizations

## Installation Requirements

Before running this notebook, ensure you have the required packages installed:

```bash
pip install pandas plotly ipywidgets
```

Or if using the notebook interface, run the cell below to install packages.

In [29]:
# Install required packages (uncomment if needed)
#!pip install pandas plotly ipywidgets

## Import Libraries

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML
from pathlib import Path
import re
from datetime import datetime


base_path = 'Other/olly-benchy/'

## Data Loading Functions

In [ ]:
def discover_result_directories(results_path='results'):
    """
    Scan the results directory and discover all benchmark run directories.
    
    Returns:
        list: List of Path objects for each result directory
    """
    results_dir = Path(results_path)
    if not results_dir.exists():
        print(f"Warning: Results directory '{results_path}' does not exist")
        return []
    
    # Get all subdirectories in results
    run_dirs = [d for d in results_dir.iterdir() if d.is_dir()]
    print(f"Found {len(run_dirs)} benchmark run(s)")
    return run_dirs


def parse_run_directory_name(dir_name):
    """
    Parse the run directory name to extract model and timestamp.
    Expected format: {model}_{timestamp}
    
    Returns:
        dict: Dictionary with 'model' and 'timestamp' keys
    """
    # Handle model names with colons (e.g., gemma4:e4b)
    parts = dir_name.rsplit('_', 1)
    if len(parts) == 2:
        model = parts[0]
        timestamp = parts[1]
    else:
        model = dir_name
        timestamp = 'unknown'
    
    return {'model': model, 'timestamp': timestamp}


def load_all_run_results(results_path='results'):
    """
    Load all run_results.csv files from all discovered benchmark runs.
    
    Returns:
        pd.DataFrame: Combined dataframe with all run results
    """
    run_dirs = discover_result_directories(results_path)
    all_results = []
    
    for run_dir in run_dirs:
        run_results_file = run_dir / 'run_results.csv'
        if run_results_file.exists():
            df = pd.read_csv(run_results_file)
            
            # Add metadata from directory name
            metadata = parse_run_directory_name(run_dir.name)
            df['run_model'] = metadata['model']
            df['run_timestamp'] = metadata['timestamp']
            df['run_directory'] = run_dir.name
            
            # Add full path to gpu_metrics_csv if it exists
            if 'gpu_metrics_csv' in df.columns:
                df['gpu_metrics_path'] = df['gpu_metrics_csv'].apply(
                    lambda x: str(run_dir / f"{x}.csv") if pd.notna(x) else None
                )
            
            all_results.append(df)
            print(f"Loaded {len(df)} tasks from {run_dir.name}")
    
    if all_results:
        combined_df = pd.concat(all_results, ignore_index=True)
        print(f"\nTotal tasks loaded: {len(combined_df)}")
        return combined_df
    else:
        print("No run results found")
        return pd.DataFrame()


def load_gpu_monitor_data(gpu_metrics_path):
    """
    Load GPU monitoring data from a CSV file.
    
    Args:
        gpu_metrics_path: Path to the GPU monitor CSV file
        
    Returns:
        pd.DataFrame: GPU monitoring data with parsed timestamps
    """
    if not gpu_metrics_path or not Path(gpu_metrics_path).exists():
        return pd.DataFrame()
    
    df = pd.read_csv(gpu_metrics_path)
    
    # Parse timestamp column
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        # Add relative time in seconds from start
        df['elapsed_seconds'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
    
    return df


def load_all_gpu_data(run_results_df):
    """
    Load all GPU monitoring data for all tasks in the run results.
    
    Args:
        run_results_df: DataFrame with run results including gpu_metrics_path
        
    Returns:
        dict: Dictionary mapping (run_directory, task_id) to GPU monitoring DataFrame
    """
    gpu_data = {}
    
    for idx, row in run_results_df.iterrows():
        if pd.notna(row.get('gpu_metrics_path')):
            key = (row['run_directory'], row['id'])
            print('---->',row['gpu_metrics_path'])
            gpu_df = load_gpu_monitor_data(row['gpu_metrics_path'])
            if not gpu_df.empty:
                # Add metadata to GPU dataframe
                gpu_df['task_id'] = row['id']
                gpu_df['task_name'] = row['name']
                gpu_df['run_directory'] = row['run_directory']
                gpu_data[key] = gpu_df
    
    print(f"Loaded GPU monitoring data for {len(gpu_data)} tasks")
    return gpu_data

## Load Data

In [ ]:
# Load all benchmark results
run_results = load_all_run_results()

# Load all GPU monitoring data
gpu_data = load_all_gpu_data(run_results)

# Display summary
if not run_results.empty:
    print(f"\nQuick Summary:")
    print(f"- Models: {run_results['run_model'].nunique()}")
    print(f"- Runs: {run_results['run_directory'].nunique()}")
    print(f"- Total Tasks: {len(run_results)}")
    print(f"- GPU Monitored Tasks: {len(gpu_data)}")
    
    # Display sample data
    display(run_results[['id', 'name', 'type', 'model', 'total_duration_s', 'tokens_per_second']].head())
else:
    print("\n  No data loaded. Please check the results directory path.")

Found 1 benchmark run(s)
Loaded 2 tasks from gemma4:e4b_20260421_144744

Total tasks loaded: 2
----> results/gemma4:e4b_20260421_144744/task_1_gpu_monitor_20260421_144810.csv
----> results/gemma4:e4b_20260421_144744/task_2_gpu_monitor_20260421_144947.csv
Loaded GPU monitoring data for 0 tasks

Quick Summary:
- Models: 1
- Runs: 1
- Total Tasks: 2
- GPU Monitored Tasks: 0


,id,name,type,model,total_duration_s,tokens_per_second
0,1,Marble,completion,gemma4:e4b,20.269554,28.564894
1,2,agent-test,agent-oneshot,gemma4:e4b,86.205146,27.672616


In [47]:
# Detailed Summary Report
print("=" * 70)
print(" " * 20 + "BENCHMARK SUMMARY REPORT")
print("=" * 70)

# Models
print("\n📊 MODELS:")
print("-" * 70)
models = run_results['run_model'].unique()
for model in sorted(models):
    model_tasks = run_results[run_results['run_model'] == model]
    print(f"  • {model:30} ({len(model_tasks)} tasks)")

# Runs
print("\n🔄 RUNS:")
print("-" * 70)
runs = run_results['run_directory'].unique()
for run in sorted(runs):
    run_tasks = run_results[run_results['run_directory'] == run]
    print(f"  • {run:50} ({len(run_tasks)} tasks)")

# Tasks
print("\n📋 TASKS:")
print("-" * 70)
tasks = run_results.groupby('name').agg({
    'id': 'count',
    'type': 'first'
}).rename(columns={'id': 'count'})
for task_name, row in tasks.iterrows():
    print(f"  • {task_name:30} [{row['type']:15}] (executed {row['count']} time(s))")

# Overall Statistics
print("\n📈 OVERALL STATISTICS:")
print("-" * 70)
print(f"  Total Models:          {len(models)}")
print(f"  Total Runs:            {len(runs)}")
print(f"  Unique Tasks:          {len(tasks)}")
print(f"  Total Executions:      {len(run_results)}")

if 'total_duration_s' in run_results.columns:
    total_time = run_results['total_duration_s'].sum()
    avg_time = run_results['total_duration_s'].mean()
    print(f"  Total Duration:        {total_time:.2f}s ({total_time/60:.2f} min)")
    print(f"  Average Task Duration: {avg_time:.2f}s")

if 'tokens_per_second' in run_results.columns:
    avg_tokens = run_results['tokens_per_second'].mean()
    print(f"  Average Tokens/Second: {avg_tokens:.2f}")

if 'tool_call_count' in run_results.columns:
    total_tool_calls = run_results['tool_call_count'].sum()
    tasks_with_tools = (run_results['tool_call_count'] > 0).sum()
    print(f"  Total Tool Calls:      {int(total_tool_calls)}")
    print(f"  Tasks Using Tools:     {tasks_with_tools}")

print("\n💾 GPU MONITORING:")
print("-" * 70)
print(f"  Tasks with GPU Data:   {len(gpu_data)}")

print("\n" + "=" * 70)

                    BENCHMARK SUMMARY REPORT

📊 MODELS:
----------------------------------------------------------------------
  • gemma4:e4b_20260421            (2 tasks)

🔄 RUNS:
----------------------------------------------------------------------
  • gemma4:e4b_20260421_144744                         (2 tasks)

📋 TASKS:
----------------------------------------------------------------------
  • Marble                         [completion     ] (executed 1 time(s))
  • agent-test                     [agent-oneshot  ] (executed 1 time(s))

📈 OVERALL STATISTICS:
----------------------------------------------------------------------
  Total Models:          1
  Total Runs:            1
  Unique Tasks:          2
  Total Executions:      2
  Total Duration:        106.47s (1.77 min)
  Average Task Duration: 53.24s
  Average Tokens/Second: 28.12
  Total Tool Calls:      3
  Tasks Using Tools:     1

💾 GPU MONITORING:
----------------------------------------------------------------------
 

## Interactive Filters

Use the dropdown menus below to filter the visualizations by model, run, or specific task.

In [34]:
# Create filter widgets
model_options = ['All'] + sorted(run_results['run_model'].unique().tolist())
run_options = ['All'] + sorted(run_results['run_directory'].unique().tolist())
task_options = ['All'] + sorted(run_results['name'].unique().tolist())

model_filter = widgets.Dropdown(
    options=model_options,
    value='All',
    description='Model:',
    style={'description_width': 'initial'}
)

run_filter = widgets.Dropdown(
    options=run_options,
    value='All',
    description='Run:',
    style={'description_width': 'initial'}
)

task_filter = widgets.Dropdown(
    options=task_options,
    value='All',
    description='Task:',
    style={'description_width': 'initial'}
)

# Display filters
filter_box = widgets.HBox([model_filter, run_filter, task_filter])
display(filter_box)

## Helper Functions for Filtering and Visualization

In [35]:
def apply_filters(df, model_val, run_val, task_val):
    """
    Apply filters to the dataframe based on widget values.
    
    Returns:
        pd.DataFrame: Filtered dataframe
    """
    filtered = df.copy()
    
    if model_val != 'All':
        filtered = filtered[filtered['run_model'] == model_val]
    
    if run_val != 'All':
        filtered = filtered[filtered['run_directory'] == run_val]
    
    if task_val != 'All':
        filtered = filtered[filtered['name'] == task_val]
    
    return filtered


def get_filtered_gpu_data(gpu_data_dict, filtered_run_results):
    """
    Get GPU data that matches the filtered run results.
    
    Returns:
        list: List of tuples (task_info, gpu_df)
    """
    filtered_gpu = []
    
    for idx, row in filtered_run_results.iterrows():
        key = (row['run_directory'], row['id'])
        if key in gpu_data_dict:
            task_info = {
                'task_id': row['id'],
                'task_name': row['name'],
                'run_directory': row['run_directory'],
                'model': row['run_model']
            }
            filtered_gpu.append((task_info, gpu_data_dict[key]))
    
    return filtered_gpu

## Performance Metrics Dashboard

Summary statistics and performance metrics for the selected tasks.

In [36]:
def create_performance_dashboard(filtered_df):
    """
    Create a performance metrics dashboard with summary cards and charts.
    """
    if filtered_df.empty:
        print("No data to display with current filters")
        return
    
    # Summary metrics
    total_tasks = len(filtered_df)
    avg_tokens_per_sec = filtered_df['tokens_per_second'].mean() if 'tokens_per_second' in filtered_df.columns else 0
    total_duration = filtered_df['total_duration_s'].sum() if 'total_duration_s' in filtered_df.columns else 0
    models_tested = filtered_df['run_model'].nunique()
    
    # Display summary cards
    print("=" * 60)
    print(f"  PERFORMANCE SUMMARY")
    print("=" * 60)
    print(f"  Total Tasks:          {total_tasks}")
    print(f"  Models Tested:        {models_tested}")
    print(f"  Avg Tokens/Second:    {avg_tokens_per_sec:.2f}")
    print(f"  Total Duration:       {total_duration:.2f}s")
    print("=" * 60)
    
    # Create subplots for visualizations
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Task Duration (seconds)',
            'Tokens per Second by Task',
            'Task Type Distribution',
            'Tool Usage Count'
        ),
        specs=[
            [{'type': 'bar'}, {'type': 'bar'}],
            [{'type': 'pie'}, {'type': 'bar'}]
        ]
    )
    
    # 1. Task Duration bar chart
    if 'total_duration_s' in filtered_df.columns:
        fig.add_trace(
            go.Bar(
                x=filtered_df['name'],
                y=filtered_df['total_duration_s'],
                name='Duration',
                marker_color='lightblue',
                showlegend=False
            ),
            row=1, col=1
        )
    
    # 2. Tokens per second bar chart
    if 'tokens_per_second' in filtered_df.columns:
        fig.add_trace(
            go.Bar(
                x=filtered_df['name'],
                y=filtered_df['tokens_per_second'],
                name='Tokens/sec',
                marker_color='lightgreen',
                showlegend=False
            ),
            row=1, col=2
        )
    
    # 3. Task type distribution pie chart
    if 'type' in filtered_df.columns:
        type_counts = filtered_df['type'].value_counts()
        fig.add_trace(
            go.Pie(
                labels=type_counts.index,
                values=type_counts.values,
                showlegend=True
            ),
            row=2, col=1
        )
    
    # 4. Tool usage bar chart
    if 'tool_call_count' in filtered_df.columns:
        tool_usage = filtered_df[filtered_df['tool_call_count'] > 0][['name', 'tool_call_count']].copy()
        if not tool_usage.empty:
            fig.add_trace(
                go.Bar(
                    x=tool_usage['name'],
                    y=tool_usage['tool_call_count'],
                    name='Tool Calls',
                    marker_color='lightcoral',
                    showlegend=False
                ),
                row=2, col=2
            )
    
    # Update layout
    fig.update_layout(
        height=800,
        title_text="Performance Metrics Overview",
        showlegend=False
    )
    
    fig.show()
    
    # Display detailed task table
    display_columns = ['id', 'name', 'type', 'total_duration_s', 'tokens_per_second', 'tool_call_count']
    available_columns = [col for col in display_columns if col in filtered_df.columns]
    
    print("\nDetailed Task Information:")
    display(filtered_df[available_columns])


# Display the dashboard
filtered_data = apply_filters(run_results, model_filter.value, run_filter.value, task_filter.value)
create_performance_dashboard(filtered_data)

  PERFORMANCE SUMMARY
  Total Tasks:          2
  Models Tested:        1
  Avg Tokens/Second:    28.12
  Total Duration:       106.47s



Detailed Task Information:


,id,name,type,total_duration_s,tokens_per_second,tool_call_count
0,1,Marble,completion,20.269554,28.564894,0
1,2,agent-test,agent-oneshot,86.205146,27.672616,3


## GPU Utilization Time-Series Graphs

Interactive time-series visualizations of GPU metrics for each task.

In [37]:
def create_gpu_timeseries_graphs(filtered_gpu_data):
    """
    Create comprehensive GPU monitoring time-series graphs for filtered tasks.
    """
    if not filtered_gpu_data:
        print("No GPU monitoring data available for current filters")
        return
    
    for task_info, gpu_df in filtered_gpu_data:
        task_label = f"{task_info['task_name']} (Task {task_info['task_id']}) - {task_info['run_directory']}"
        
        # Create subplots for different GPU metrics
        fig = make_subplots(
            rows=4, cols=1,
            subplot_titles=(
                f'{task_label} - GPU Utilization',
                'Memory Utilization',
                'Power Consumption',
                'Temperature'
            ),
            vertical_spacing=0.08
        )
        
        # 1. GPU Utilization
        if 'gpu_util_percent' in gpu_df.columns:
            fig.add_trace(
                go.Scatter(
                    x=gpu_df['elapsed_seconds'],
                    y=gpu_df['gpu_util_percent'],
                    mode='lines',
                    name='GPU Util %',
                    line=dict(color='blue', width=2),
                    fill='tozeroy',
                    fillcolor='rgba(0, 0, 255, 0.1)'
                ),
                row=1, col=1
            )
        
        # 2. Memory Utilization
        if 'memory_util_percent' in gpu_df.columns:
            fig.add_trace(
                go.Scatter(
                    x=gpu_df['elapsed_seconds'],
                    y=gpu_df['memory_util_percent'],
                    mode='lines',
                    name='Memory Util %',
                    line=dict(color='green', width=2),
                    fill='tozeroy',
                    fillcolor='rgba(0, 255, 0, 0.1)'
                ),
                row=2, col=1
            )
        
        # 3. Power Consumption
        if 'power_watts' in gpu_df.columns:
            fig.add_trace(
                go.Scatter(
                    x=gpu_df['elapsed_seconds'],
                    y=gpu_df['power_watts'],
                    mode='lines',
                    name='Power (W)',
                    line=dict(color='orange', width=2),
                    fill='tozeroy',
                    fillcolor='rgba(255, 165, 0, 0.1)'
                ),
                row=3, col=1
            )
        
        # 4. Temperature
        if 'temperature_c' in gpu_df.columns:
            fig.add_trace(
                go.Scatter(
                    x=gpu_df['elapsed_seconds'],
                    y=gpu_df['temperature_c'],
                    mode='lines',
                    name='Temperature (°C)',
                    line=dict(color='red', width=2),
                    fill='tozeroy',
                    fillcolor='rgba(255, 0, 0, 0.1)'
                ),
                row=4, col=1
            )
        
        # Update axes labels
        fig.update_xaxes(title_text="Time (seconds)", row=4, col=1)
        fig.update_yaxes(title_text="GPU Util (%)", row=1, col=1)
        fig.update_yaxes(title_text="Memory Util (%)", row=2, col=1)
        fig.update_yaxes(title_text="Power (W)", row=3, col=1)
        fig.update_yaxes(title_text="Temperature (°C)", row=4, col=1)
        
        # Update layout
        fig.update_layout(
            height=1000,
            showlegend=False,
            title_text=f"GPU Monitoring - {task_label}"
        )
        
        fig.show()
        
        # Display summary statistics
        print(f"\n{task_label} - GPU Statistics:")
        print("-" * 60)
        
        metrics = {
            'GPU Util (%)': 'gpu_util_percent',
            'Memory Util (%)': 'memory_util_percent',
            'Power (W)': 'power_watts',
            'Temperature (°C)': 'temperature_c'
        }
        
        for metric_name, col_name in metrics.items():
            if col_name in gpu_df.columns:
                print(f"{metric_name:20} - Min: {gpu_df[col_name].min():6.2f}  "
                      f"Max: {gpu_df[col_name].max():6.2f}  "
                      f"Avg: {gpu_df[col_name].mean():6.2f}")
        
        print("-" * 60)
        print()


# Display GPU time-series graphs
filtered_data = apply_filters(run_results, model_filter.value, run_filter.value, task_filter.value)
filtered_gpu = get_filtered_gpu_data(gpu_data, filtered_data)
create_gpu_timeseries_graphs(filtered_gpu)

No GPU monitoring data available for current filters


## Comparison Visualizations

Aggregate views and comparisons across tasks and runs.

In [27]:
def create_comparison_visualizations(filtered_gpu_data):
    """
    Create aggregate comparison visualizations across tasks.
    """
    if not filtered_gpu_data:
        print("No GPU monitoring data available for comparisons")
        return
    
    # Aggregate statistics for each task
    task_stats = []
    
    for task_info, gpu_df in filtered_gpu_data:
        stats = {
            'Task': f"{task_info['task_name']} (ID: {task_info['task_id']})",
            'Run': task_info['run_directory'],
        }
        
        # Calculate statistics for each metric
        metrics = {
            'GPU Util (%)': 'gpu_util_percent',
            'Memory Util (%)': 'memory_util_percent',
            'Power (W)': 'power_watts',
            'Temperature (°C)': 'temperature_c'
        }
        
        for metric_name, col_name in metrics.items():
            if col_name in gpu_df.columns:
                stats[f'{metric_name} - Avg'] = gpu_df[col_name].mean()
                stats[f'{metric_name} - Max'] = gpu_df[col_name].max()
        
        task_stats.append(stats)
    
    stats_df = pd.DataFrame(task_stats)
    
    # Create comparison visualizations
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Average GPU Utilization by Task',
            'Average Memory Utilization by Task',
            'Average Power Consumption by Task',
            'Maximum Temperature by Task'
        )
    )
    
    # 1. Average GPU Utilization
    if 'GPU Util (%) - Avg' in stats_df.columns:
        fig.add_trace(
            go.Bar(
                x=stats_df['Task'],
                y=stats_df['GPU Util (%) - Avg'],
                name='Avg GPU Util',
                marker_color='blue',
                showlegend=False
            ),
            row=1, col=1
        )
    
    # 2. Average Memory Utilization
    if 'Memory Util (%) - Avg' in stats_df.columns:
        fig.add_trace(
            go.Bar(
                x=stats_df['Task'],
                y=stats_df['Memory Util (%) - Avg'],
                name='Avg Memory Util',
                marker_color='green',
                showlegend=False
            ),
            row=1, col=2
        )
    
    # 3. Average Power Consumption
    if 'Power (W) - Avg' in stats_df.columns:
        fig.add_trace(
            go.Bar(
                x=stats_df['Task'],
                y=stats_df['Power (W) - Avg'],
                name='Avg Power',
                marker_color='orange',
                showlegend=False
            ),
            row=2, col=1
        )
    
    # 4. Maximum Temperature
    if 'Temperature (°C) - Max' in stats_df.columns:
        fig.add_trace(
            go.Bar(
                x=stats_df['Task'],
                y=stats_df['Temperature (°C) - Max'],
                name='Max Temp',
                marker_color='red',
                showlegend=False
            ),
            row=2, col=2
        )
    
    # Update axes
    fig.update_yaxes(title_text="GPU Util (%)", row=1, col=1)
    fig.update_yaxes(title_text="Memory Util (%)", row=1, col=2)
    fig.update_yaxes(title_text="Power (W)", row=2, col=1)
    fig.update_yaxes(title_text="Temperature (°C)", row=2, col=2)
    
    # Update layout
    fig.update_layout(
        height=800,
        title_text="GPU Metrics Comparison Across Tasks",
        showlegend=False
    )
    
    fig.show()
    
    # Display summary statistics table
    print("\nSummary Statistics Table:")
    display(stats_df)


# Display comparison visualizations
filtered_data = apply_filters(run_results, model_filter.value, run_filter.value, task_filter.value)
filtered_gpu = get_filtered_gpu_data(gpu_data, filtered_data)
create_comparison_visualizations(filtered_gpu)

No GPU monitoring data available for comparisons


## Interactive Update

Run the cell below to update all visualizations based on the current filter selections.

In [28]:
def update_all_visualizations():
    """
    Update all visualizations based on current filter selections.
    Re-run this cell after changing filter values.
    """
    print("Updating visualizations with current filters...")
    print(f"Model: {model_filter.value}")
    print(f"Run: {run_filter.value}")
    print(f"Task: {task_filter.value}")
    print("\n" + "="*60 + "\n")
    
    # Apply filters
    filtered_data = apply_filters(run_results, model_filter.value, run_filter.value, task_filter.value)
    filtered_gpu = get_filtered_gpu_data(gpu_data, filtered_data)
    
    # Update all visualizations
    print("1. PERFORMANCE DASHBOARD")
    print("="*60)
    create_performance_dashboard(filtered_data)
    
    print("\n\n2. GPU TIME-SERIES GRAPHS")
    print("="*60)
    create_gpu_timeseries_graphs(filtered_gpu)
    
    print("\n\n3. COMPARISON VISUALIZATIONS")
    print("="*60)
    create_comparison_visualizations(filtered_gpu)
    
    print("\n" + "="*60)
    print("✓ All visualizations updated!")


# Create update button
update_button = widgets.Button(
    description='Update Visualizations',
    button_style='success',
    tooltip='Click to update all visualizations with current filter settings',
    icon='refresh'
)

def on_update_button_clicked(b):
    update_all_visualizations()

update_button.on_click(on_update_button_clicked)

display(update_button)

print("\nTo update visualizations:")
print("1. Select filters from the dropdowns above")
print("2. Click the 'Update Visualizations' button")
print("3. Or re-run this cell manually")

Button(button_style='success', description='Update Visualizations', icon='refresh', style=ButtonStyle(), toolt…


To update visualizations:
1. Select filters from the dropdowns above
2. Click the 'Update Visualizations' button
3. Or re-run this cell manually


,id,name,type,total_duration_s,tokens_per_second,tool_call_count
0,1,Marble,completion,20.269554,28.564894,0
1,2,agent-test,agent-oneshot,86.205146,27.672616,3
